In [10]:
from ollama import chat
import pandas as pd
import json

MODEL = "llama3.2"

In [12]:
CSV_PATH = "db2.csv"

wanted = ["email_text", "gold_reason"]
df = pd.read_csv(CSV_PATH, usecols=wanted)

df.columns = [c.strip() for c in df.columns]

df["email_text"] = df["email_text"].astype(str).str.strip()
df["gold_reason"] = (
    df["gold_reason"].astype(str).str.lower().str.strip().replace({"": None, "nan": None})
)

df.head()

,email_text,gold_reason
0,"I absolutely love the new dashboard, great imp...",clear positive tone
1,"New pricing page loaded weirdly, idk what changed","missing details, ambiguous tone"
2,The update looks okay but I have nothing else ...,"minimal cue, weak justification"
3,Pricing increase is frustrating and unnecessary,strong negative wording
4,The agent was helpful and resolved my issue qu...,resolved issue and satisfaction


In [13]:
SYSTEM_MSG = """
You are a customer sentiment analysis assistant.

For every email, return ONLY valid JSON.

For each email id return:

{
  "A": {
    "sentiment": "positive|negative|neutral",
    "confidence": 0.0,
    "reason": "exact quote"
  },
  "B": {
    "sentiment": "positive|negative|neutral",
    "confidence": 0.0,
    "reason": "one or two exact quotes"
  },
  "judge": {
    "winner": "A|B|same",
    "explanation": "brief explanation"
  }
}

Return ONE JSON object whose keys are the email ids.
Do not include markdown.
Do not include any text outside the JSON.
""".strip()


def build_batched_prompt(email_dict):
    prompt = "Evaluate the following emails.\n\n"

    for eid, text in email_dict.items():
        clean = text.replace("\n", " ")
        prompt += f'Email ID: {eid}\nEmail: "{clean}"\n\n'

    return prompt

In [14]:
from ollama import chat
import json

def call_gpt_json(prompt, model=MODEL):

    response = chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_MSG
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    content = response["message"]["content"].strip()

    # Try parsing directly
    try:
        return json.loads(content)

    # Otherwise extract the first JSON object
    except:
        try:
            start = content.index("{")
            end = content.rindex("}") + 1
            return json.loads(content[start:end])
        except Exception:
            print("Model returned invalid JSON:\n")
            print(content)
            return {}


def run_batched_llm(df, batch_size=20):

    all_results = {}

    for start in range(0, len(df), batch_size):

        chunk = df.iloc[start:start + batch_size]

        emails = {
            str(i): str(row["email_text"])
            for i, row in chunk.iterrows()
        }

        prompt = build_batched_prompt(emails)

        batch_results = call_gpt_json(prompt)

        all_results.update(batch_results)

    return all_results

In [16]:
def results_to_dataframe(results_dict, df):
    rows = []

    for eid, content in results_dict.items():
        A = content.get("A", {})
        B = content.get("B", {})
        J = content.get("judge", {})

        email_text = df.loc[int(eid), "email_text"]
        gold = df.loc[int(eid), "gold_reason"] if "gold_reason" in df.columns else None

        A_conf = float(A.get("confidence", 0) or 0)
        B_conf = float(B.get("confidence", 0) or 0)

        # --- PER EMAIL DECISION (your requirement) ---
        if A_conf > B_conf:
            final_label = A.get("sentiment")
            selected = "A"
        elif B_conf > A_conf:
            final_label = B.get("sentiment")
            selected = "B"
        else:
            final_label = A.get("sentiment")  # tie → choose A
            selected = "same_confidence"

        rows.append({
            "id": eid,
            "email": email_text,
            "gold_reason": gold,

            "A_sentiment": A.get("sentiment"),
            "A_confidence": A_conf,
            "A_reason": A.get("reason"),

            "B_sentiment": B.get("sentiment"),
            "B_confidence": B_conf,
            "B_reason": B.get("reason"),

            "judge_suggested": J.get("suggested_sentiment"),
            "judge_difference": J.get("difference"),
            "judge_explanation": J.get("explanation"),

            # FINAL result (your requirement)
            "final_sentiment": final_label,
            "selected_by": selected
        })

    return pd.DataFrame(rows)

In [18]:
batched_results = run_batched_llm(df, batch_size=20)
results_df = results_to_dataframe(batched_results, df)
results_df.head()

Model returned invalid JSON:

{
  "0": {
    "A": {
      "sentiment": "positive",
      "confidence": 1.0,
      "reason": ""
    },
    "B": {
      "sentiment": "neutral",
      "confidence": 1.0,
      "reason": ""
    },
    "judge": {
      "winner": "A",
      "explanation": ""
    }
  },
  "1": {
    "A": {
      "sentiment": "negative",
      "confidence": 1.0,
      "reason": ""
    },
    "B": {
      "sentiment": "neutral",
      "confidence": 1.0,
      "reason": ""
    },
    "judge": {
      "winner": "A",
      "explanation": ""
    }
  },
  "2": {
    "A": {
      "sentiment": "negative",
      "confidence": 1.0,
      "reason": ""
    },
    "B": {
      "sentiment": "neutral",
      "confidence": 1.0,
      "reason": ""
    },
    "judge": {
      "winner": "A",
      "explanation": ""
    }
  },
  "3": {
    "A": {
      "sentiment": "negative",
      "confidence": 1.0,
      "reason": ""
    },
    "B": {
      "sentiment": "positive",
      "confidence": 1.0,
    

,id,email,gold_reason,A_sentiment,A_confidence,A_reason,B_sentiment,B_confidence,B_reason,judge_suggested,judge_difference,judge_explanation,final_sentiment,selected_by
0,20,"I think the system is more stable now, fewer e...",measurable improvement,positive,0.8,exact quote,neutral,0.2,incomplete sentence,None,None,Email 20 has a clear positive sentiment from t...,positive,A
1,21,Not sure if the system changed or maybe I just...,uncertain/self-awareness,neutral,0.5,incomplete sentence,negative,0.3,possible negative tone implied by 'maybe I jus...,None,None,Email 21 has a neutral sentiment from the inco...,neutral,A
2,22,"UI update seems fine, honestly not a big chang...",minimal perceived impact,neutral,0.4,incomplete phrase,negative,0.5,possible negative tone implied by 'not a big c...,None,None,Email 22 has a neutral sentiment from the inco...,negative,B
3,23,"After this update I find everything faster, th...",speed improvement appreciation,positive,0.9,exact quote,neutral,0.1,incomplete phrase,None,None,Email 23 has a clear positive sentiment from t...,positive,A
4,24,"The new workflow is cleaner, onboarding was ea...",lower onboarding friction,positive,0.8,exact quote,neutral,0.2,incomplete phrase,None,None,Email 24 has a clear positive sentiment from t...,positive,A


In [19]:
def add_review_flags(results_df, low_conf=0.6):
    def needs_review(row):
        if row["A_confidence"] < low_conf and row["B_confidence"] < low_conf:
            return "Both confidences low"
        if row["A_sentiment"] != row["B_sentiment"]:
            return "A and B disagree"
        return None

    results_df["review_reason"] = results_df.apply(needs_review, axis=1)
    results_df["needs_review"] = results_df["review_reason"].notnull()
    return results_df

results_df = add_review_flags(results_df)
results_df[["id", "final_sentiment", "needs_review", "review_reason"]].head()

,id,final_sentiment,needs_review,review_reason
0,20,positive,True,A and B disagree
1,21,neutral,True,Both confidences low
2,22,negative,True,Both confidences low
3,23,positive,True,A and B disagree
4,24,positive,True,A and B disagree


In [20]:
def export_human_review_queue(results_df, out_path="human_review_queue.csv"):
    review_df = results_df[results_df["needs_review"]].copy()

    cols = [
        "id",
        "email",
        "final_sentiment",
        "A_sentiment", "A_confidence",
        "B_sentiment", "B_confidence",
        "review_reason"
    ]
    cols = [c for c in cols if c in review_df.columns]

    review_df[cols].to_csv(out_path, index=False)
    return out_path

review_path = export_human_review_queue(results_df)
print("Human review file created:", review_path)

Human review file created: human_review_queue.csv


In [21]:
def apply_human_review(results_df, reviewed_csv="human_reviewed.csv"):
    reviewed = pd.read_csv(reviewed_csv)

    reviewed = reviewed[["id", "human_final_sentiment", "human_notes"]]

    merged = results_df.merge(reviewed, on="id", how="left")

    # If human provided a decision, override model output
    merged["final_sentiment"] = merged["human_final_sentiment"].fillna(
        merged["final_sentiment"]
    )

    merged["review_source"] = merged["human_final_sentiment"].apply(
        lambda x: "human" if pd.notnull(x) else "model"
    )

    return merged

In [22]:
from datetime import datetime
import os

def export_final_approved(results_df, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%dT%H%M%S")
    path = os.path.join(out_dir, f"final_approved_results_{ts}.csv")

    cols = [
        "email",
        "final_sentiment",
        "review_source",
        "A_confidence",
        "B_confidence",
    ]
    cols = [c for c in cols if c in results_df.columns]

    results_df[cols].to_csv(path, index=False)
    return path


final_path = export_final_approved(results_df)
print("Final approved file saved at:", final_path)

Final approved file saved at: outputs\final_approved_results_20260703T045304.csv


In [23]:
if "review_source" in results_df.columns:
    new_gold = results_df[results_df["review_source"] == "human"][
        ["email", "final_sentiment"]
    ]
else:
    print("No human reviews yet — nothing to extract as new gold.")

No human reviews yet — nothing to extract as new gold.
